# Índice de incertidumbre de política comercial **sobre México** con GDELT

Este notebook construye desde cero un índice al estilo de Caldara, Iacoviello et al. (TPU), con un filtro adicional: **el artículo debe mencionar a México**. Usa la API DOC 2.0 de GDELT, que indexa el texto completo de noticias en línea de todo el mundo desde enero de 2017 (los artículos en otros idiomas se buscan en su traducción automática al inglés).

**Se ejecuta en su computadora** (requiere internet). La descarga completa tarda entre 1.5 y 3 horas por el límite de GDELT (una solicitud cada 5 segundos). Cada respuesta se guarda en disco: si se interrumpe, vuelva a ejecutar y continúa donde se quedó.

## Lógica

**1. Qué se cuenta.** Para cada mes, el número de artículos que cumplen **las tres condiciones a la vez**:

| Grupo | Caldara et al. (ProQuest) | Traducción a GDELT |
|---|---|---|
| Tema comercial | "tariff\*", "import duty", "import barrier", "trade treaty", "trade policy", "trade act", dumping, "import fee", tax n/10 ("foreign goods" OR "foreign oil" OR import\*), (imports OR import OR imported) n/10 (surtax OR surcharge\*) | GDELT no tiene comodín `*`: se listan las variantes (tariff, tariffs; "import duties"…). La cercanía `n/10` se traduce al operador de GDELT `near10:"palabra1 palabra2"`. |
| Incertidumbre | uncertainty, uncertain, risk\*, dubious, unclear, potential, probabl\*, predict\*, danger\* | Se listan las variantes de cada raíz (risk, risks, risky, risked…). «unpredictable» **no** se incluye: no empieza con «predict». |
| **México (filtro nuevo)** | — | Mexico, Mexican, Mexicans (GDELT busca palabras completas: «Mexico» no encuentra «Mexican»). |

**Límite de GDELT.** La API rechaza consultas largas («Your query was too short or too long»). En pruebas del 1 de octubre de 2026 rechazó la consulta completa (734 caracteres) y versiones de 541 y 361, y aceptó una de 204. El límite está entre 230 y 361 caracteres, así que **no caben todos los términos de Caldara et al.** El notebook tiene **seis versiones**, de más a menos completa (completa → sin cercanía → compacta → intermedia → reducida → mínima), y usa **la más completa que GDELT acepte**, la misma en todas las series. Se conservan los términos más frecuentes («tariff(s)», «trade policy», «risk(s)», «uncertainty»…) y se mide cuánto podrían aportar los omitidos. La validación contra el TPU original (sección 8) dice si la versión reducida sigue captando el fenómeno.

**2. De qué medios.** Se construyen tres versiones, porque GDELT permite filtrar por país del medio (`sourcecountry`):

- **EE.UU.:** medios estadounidenses. Es la más cercana al TPU original, que usa prensa de EE.UU. **Recomendada como principal.**
- **México:** medios mexicanos (en español, buscados en su traducción). Casi todos sus artículos mencionan a México, así que ahí el filtro casi no actúa.
- **Global:** todos los medios que monitorea GDELT.

**3. Contra qué se normaliza.** Igual que Caldara (artículos que cumplen / total de artículos × 10,000; 100 = 1% de los artículos), con dos denominadores:

- **Total de artículos de los medios del mismo país** (consulta solo con `sourcecountry`). Es el equivalente directo de `N7`.
- **Total de artículos monitoreados por GDELT** (campo `norm` de la API). Se usa en la versión global.

GDELT ha ido agregando medios con el tiempo; normalizar por el total corrige ese crecimiento.

**4. Validación.** Se construye además la versión **sin** el filtro de México con medios de EE.UU. (el equivalente GDELT del TPU) y se compara con el TPU de Caldara et al. que ya tenemos. Si el método funciona, deben moverse juntos.

**5. Diferencias que no se pueden evitar.** Son noticias en línea, no los 7 periódicos impresos; la cobertura empieza en 2017; GDELT busca en la traducción automática de textos no ingleses, y cuenta notas repetidas por sindicación. Por eso el índice es **una aproximación al método**, no una réplica exacta.

## 0. Configuración
Ajuste solo este bloque si hace falta. **Todo se guarda en la carpeta donde está trabajando** (la carpeta desde la que abre el notebook): ahí se crean `gdelt_cache/`, el CSV y los metadatos. No necesita el repositorio.

Requisitos: `pip install pandas numpy requests openpyxl matplotlib truststore`

**Errores SSL.** `truststore` hace que Python use los certificados de su sistema operativo (como el navegador). Así se evitan los `SSLError` comunes en Windows, en redes universitarias o de empresa y con antivirus que revisan HTTPS. Si aun así falla HTTPS, con `PERMITIR_HTTP = True` el notebook pasa a `http://`. GDELT es un servicio público de solo lectura y no se envían credenciales, así que el riesgo es mínimo. El protocolo que se usó queda en los metadatos.

In [ ]:
import json, time, re, calendar, hashlib, platform, shutil, subprocess
from datetime import datetime, date
from pathlib import Path

import numpy as np
import pandas as pd
import requests
try:
    import truststore; truststore.inject_into_ssl()   # usar los certificados del sistema operativo
    _TRUSTSTORE = True
except ImportError:
    _TRUSTSTORE = False

INICIO = "2017-01"          # primer mes del archivo de GDELT DOC 2.0
FIN = None                  # None = último mes completo
PAUSA_SEG = 6.0             # GDELT pide una solicitud cada 5 segundos como máximo
MAX_REINTENTOS = 8
TIMEOUT_SEG = 120
URL = "https://api.gdeltproject.org/api/v2/doc/doc"
METODO = None              # None = la prueba de conexión elige; o fije ("curl", "https"), ("requests", "https")…
PERMITIR_HTTP = True        # si HTTPS falla por certificado (SSLError), usar http:// (datos públicos, sin credenciales)
NIVEL = None                # None = el diagnóstico elige la versión más completa que GDELT acepte

CARPETA = Path.cwd().resolve()   # carpeta de trabajo: aquí se guarda todo
CACHE = CARPETA / "gdelt_cache"
SALIDA = CARPETA
CACHE.mkdir(parents=True, exist_ok=True)
URL_CALDARA = "https://www.matteoiacoviello.com/tpu_files/tpu_web_latest.xlsx"
hoy = date.today()
if FIN is None:
    FIN = (pd.Timestamp(hoy.year, hoy.month, 1) - pd.offsets.MonthBegin(1)).strftime("%Y-%m")
MESES = pd.period_range(INICIO, FIN, freq="M")
print(f"Certificados del sistema (truststore): {'sí' if _TRUSTSTORE else 'no — si aparece SSLError: pip install truststore'}")
print(f"Carpeta de trabajo: {CARPETA}\nCaché: {CACHE}\nMeses: {MESES[0]} a {MESES[-1]} ({len(MESES)})")

## 1. Términos y consultas
Las listas reproducen los términos de Caldara et al. Cada cambio de traducción queda a la vista en la tabla.

In [ ]:
# GDELT rechaza consultas largas («Your query was too short or too long»). En pruebas de
# octubre de 2026 rechazó 734, 541 y 361 caracteres y aceptó 204 (versión «minimo»). Por eso hay
# seis versiones, de más a menos completa; el diagnóstico (sección 4) usa la primera que GDELT
# acepte, y la misma para todas las series. Los términos que se conservan son los más frecuentes.
COMERCIO_COMPLETO = ['tariff', 'tariffs', '"import duty"', '"import duties"', '"import barrier"', '"import barriers"',
                     '"trade treaty"', '"trade treaties"', '"trade policy"', '"trade policies"', '"trade act"',
                     'dumping', '"import fee"', '"import fees"']
# tax n/10 ("foreign goods" OR "foreign oil" OR import*)  y  (imports OR import OR imported) n/10 (surtax OR surcharge*)
CERCANIA = ['near10:"tax foreign goods"', 'near10:"tax foreign oil"', 'near10:"tax imports"', 'near10:"tax imported"',
            'near10:"imports surcharge"', 'near10:"import surcharge"', 'near10:"imports surtax"']
INCERTIDUMBRE_COMPLETA = ['uncertainty', 'uncertainties', 'uncertain', 'risk', 'risks', 'risky', 'dubious', 'unclear',
                          'potential', 'potentially', 'probable', 'probably', 'probability', 'predict', 'predicted',
                          'prediction', 'predictions', 'predictable', 'danger', 'dangers', 'dangerous']
NIVELES = {
    "completo":     dict(comercio=COMERCIO_COMPLETO, cercania=CERCANIA, incertidumbre=INCERTIDUMBRE_COMPLETA),
    "sin_cercania": dict(comercio=COMERCIO_COMPLETO, cercania=[], incertidumbre=INCERTIDUMBRE_COMPLETA),
    "compacto":     dict(comercio=['tariff', 'tariffs', '"import duty"', '"import duties"', '"import barrier"', '"trade treaty"',
                                   '"trade policy"', '"trade act"', 'dumping', '"import fee"'], cercania=[],
                         incertidumbre=['uncertainty', 'uncertain', 'risk', 'risks', 'risky', 'dubious', 'unclear', 'potential',
                                        'probably', 'probable', 'predict', 'predicted', 'danger', 'dangerous']),
    "intermedio":   dict(comercio=['tariff', 'tariffs', '"trade policy"', '"import duty"', '"import duties"', 'dumping', '"trade act"'], cercania=[],
                         incertidumbre=['uncertainty', 'uncertain', 'risk', 'risks', 'unclear', 'potential', 'probably', 'predict', 'danger', 'dangerous']),
    "reducido":     dict(comercio=['tariff', 'tariffs', '"trade policy"', '"import duty"', 'dumping', '"trade act"'], cercania=[],
                         incertidumbre=['uncertainty', 'uncertain', 'risk', 'risks', 'unclear', 'potential', 'probably', 'danger', 'predict']),
    "minimo":       dict(comercio=['tariff', 'tariffs', '"trade policy"', '"import duty"', 'dumping'], cercania=[],
                         incertidumbre=['uncertainty', 'uncertain', 'risk', 'risks', 'unclear', 'potential', 'probably', 'danger']),
}
MEXICO = ['Mexico', 'Mexican', 'Mexicans']

def grupo(terminos):
    return "(" + " OR ".join(terminos) + ")"

def consulta(con_mexico=True, pais=None, nivel=None):
    n = NIVELES[nivel or NIVEL or "completo"]
    partes = [grupo(n["comercio"] + n["cercania"]), grupo(n["incertidumbre"])]
    if con_mexico:
        partes.append(grupo(MEXICO))
    if pais:
        partes.append(f"sourcecountry:{pais}")
    return " ".join(partes)

display(pd.DataFrame([dict(version=k, terminos_comercio=len(v["comercio"]), clausulas_cercania=len(v["cercania"]),
                           terminos_incertidumbre=len(v["incertidumbre"]), longitud_consulta=len(consulta(True, "US", k)))
                      for k, v in NIVELES.items()]))
pd.set_option("display.max_colwidth", 200)
display(pd.DataFrame([
    ("Tema comercial", 'tariff*', "tariff, tariffs"),
    ("Tema comercial", '"import duty", "import barrier", "trade treaty", "trade policy", "import fee"', "frase en singular y plural"),
    ("Tema comercial", 'tax n/10 ("foreign goods" OR "foreign oil" OR import*)', 'near10:"tax foreign goods", near10:"tax imports"… (4 cláusulas)'),
    ("Tema comercial", "(imports OR import OR imported) n/10 (surtax OR surcharge*)", 'near10:"imports surcharge"… (3 cláusulas)'),
    ("Incertidumbre", "risk*, probabl*, predict*, danger*", "variantes listadas (sin «unpredictable»)"),
    ("México", "(nuevo)", "Mexico OR Mexican OR Mexicans"),
], columns=["grupo", "Caldara et al.", "GDELT"]))
print("Consulta principal, versión completa (medios de EE.UU.):\n", consulta(True, "US", "completo"))

## 2. Series que se descargan
Cada serie es una consulta que se hace **mes por mes** (una solicitud por mes; GDELT devuelve conteos diarios que luego se suman). Los denominadores por país (consultas solo con `sourcecountry`) se agregan en la sección 4 **si el diagnóstico confirma que GDELT los acepta**.

In [ ]:
def construir_series():
    return {
        "tpu_mx_eeuu":   dict(q=consulta(True, "US"),  desc="TPU + México, medios de EE.UU. (principal)"),
        "tpu_mx_mexico": dict(q=consulta(True, "MX"),  desc="TPU + México, medios mexicanos"),
        "tpu_mx_global": dict(q=consulta(True, None),  desc="TPU + México, todos los medios"),
        "tpu_eeuu":      dict(q=consulta(False, "US"), desc="TPU sin filtro de México, medios de EE.UU. (validación contra Caldara)"),
    }
SERIES = construir_series()
DENOMINADORES = {
    "total_eeuu":    dict(q="sourcecountry:US", desc="Denominador: todos los artículos de medios de EE.UU."),
    "total_mexico":  dict(q="sourcecountry:MX", desc="Denominador: todos los artículos de medios mexicanos"),
}
display(pd.DataFrame([(k, v["desc"], len(v["q"])) for k, v in SERIES.items()], columns=["serie", "descripción", "longitud de la consulta"]))
n_sol = (len(SERIES) + len(DENOMINADORES)) * len(MESES)
print(f"Solicitudes necesarias (con denominadores por país): {n_sol} (≈ {n_sol * PAUSA_SEG / 3600:.1f} horas sin contar reintentos)")

## 3. Cliente de la API con caché, pausa y reintentos
- Cada respuesta se guarda en `gdelt_cache/<serie>/<AAAAMM>.json` (en la carpeta de trabajo) junto con la consulta exacta y la fecha de descarga. Si el archivo existe y la consulta no cambió, no se vuelve a pedir.
- Se espera `PAUSA_SEG` entre solicitudes. Ante un 429 (demasiadas solicitudes) o un error del servidor, se espera cada vez más (30 s, 60 s, …).
- **Prueba de conexión.** Antes de descargar se prueban, en orden, cuatro formas de conectarse: Python por HTTPS, `curl` por HTTPS, Python por HTTP y `curl` por HTTP (las dos últimas solo con `PERMITIR_HTTP = True`). Se usa la primera que devuelve una respuesta válida de GDELT. `curl` viene incluido en Windows 10/11 y macOS, y usa los certificados del sistema. Si una forma deja de funcionar a media descarga, se vuelve a probar y se cambia de forma.
- GDELT responde con **texto plano** cuando la consulta tiene un problema (por ejemplo, una frase demasiado corta o común). En ese caso el proceso se detiene y muestra el mensaje, en lugar de guardar un resultado vacío.

In [ ]:
_ultima = [0.0]
def _esperar():
    falta = PAUSA_SEG - (time.time() - _ultima[0])
    if falta > 0:
        time.sleep(falta)
    _ultima[0] = time.time()

def _por_requests(url, params):
    r = requests.get(url, params=params, timeout=TIMEOUT_SEG)
    return r.status_code, r.text

def _por_curl(url, params):
    cmd = ["curl", "-sS", "-G", "--max-time", str(int(TIMEOUT_SEG)), "-w", "\n%{http_code}", url]
    for k, v in params.items():
        cmd += ["--data-urlencode", f"{k}={v}"]
    res = subprocess.run(cmd, capture_output=True, timeout=TIMEOUT_SEG + 30)
    if res.returncode != 0:
        raise ConnectionError(f"curl salió con código {res.returncode}: {res.stderr.decode('utf-8', 'replace').strip()[:300]}")
    texto, _, codigo = res.stdout.decode("utf-8", "replace").rpartition("\n")
    return int(codigo), texto

def _solicitud(metodo, params):
    cliente, protocolo = metodo
    url = URL.replace("https://", f"{protocolo}://", 1)
    return (_por_curl if cliente == "curl" else _por_requests)(url, params)

def probar_conexion():
    """Prueba cada forma de conexión con una consulta mínima y fija METODO en la primera que funciona."""
    global METODO
    candidatos = [("requests", "https")] + ([("curl", "https")] if shutil.which("curl") else [])
    if PERMITIR_HTTP:
        candidatos += [("requests", "http")] + ([("curl", "http")] if shutil.which("curl") else [])
    prueba = {"query": "tariff", "mode": "timelinevolraw", "format": "json",
              "startdatetime": "20190530000000", "enddatetime": "20190531235959"}
    filas = []
    for m in candidatos:
        estado = None
        for intento in range(3):
            _esperar()
            try:
                codigo, texto = _solicitud(m, prueba)
            except Exception as e:
                estado = f"falla: {e.__class__.__name__}: {str(e)[:160]}"; break
            if codigo == 429:
                estado = "HTTP 429 (límite de GDELT); se reintenta"; time.sleep(20); continue
            try:
                ok = "timeline" in json.loads(texto)
                estado = "OK" if ok else f"respuesta sin datos: {texto[:120]!r}"
            except json.JSONDecodeError:
                estado = f"respuesta inválida (HTTP {codigo}): {texto[:120]!r}"
            break
        filas.append((f"{m[0]} + {m[1]}", estado))
        if estado == "OK":
            METODO = m
            break
    display(pd.DataFrame(filas, columns=["forma de conexión", "resultado"]))
    if METODO is None:
        raise RuntimeError(
            "Ninguna forma de conexión obtuvo una respuesta válida de GDELT.\n"
            "Causas habituales: un antivirus con análisis de HTTPS/web (ESET, Kaspersky, Avast, Bitdefender…), "
            "la red de la universidad o empresa, o una VPN.\nPruebe: (1) desactivar temporalmente el análisis web del antivirus; "
            "(2) otra red, por ejemplo el hotspot del celular; (3) abrir en el navegador:\n"
            "https://api.gdeltproject.org/api/v2/doc/doc?query=tariff&mode=timelinevolraw&format=json&timespan=1d")
    print(f"Se usará: {METODO[0]} por {METODO[1].upper()}")

def pedir(query, inicio, fin, mode="timelinevolraw", extra=None):
    params = {"query": query, "mode": mode, "format": "json",
              "startdatetime": inicio.strftime("%Y%m%d%H%M%S"), "enddatetime": fin.strftime("%Y%m%d%H%M%S")}
    if extra:
        params.update(extra)
    global METODO
    fallas_seguidas = 0
    for k in range(MAX_REINTENTOS):
        _esperar()
        try:
            codigo, texto = _solicitud(METODO, params)
        except Exception as e:
            fallas_seguidas += 1
            print(f"  error de red con {METODO[0]}+{METODO[1]} ({e.__class__.__name__}: {str(e)[:200]})")
            if fallas_seguidas >= 2:                # la forma de conexión dejó de servir: se vuelve a elegir
                METODO = None; probar_conexion(); fallas_seguidas = 0
            else:
                espera = 30 * (k + 1); print(f"  reintento en {espera} s"); time.sleep(espera)
            continue
        fallas_seguidas = 0
        if codigo == 429 or codigo >= 500:
            espera = 30 * (k + 1); print(f"  HTTP {codigo}; reintento en {espera} s"); time.sleep(espera); continue
        texto = texto.strip()
        if not texto:
            return {}                       # sin resultados
        try:
            return json.loads(texto)
        except json.JSONDecodeError:
            if not texto[:1].isprintable():  # bytes basura: respuesta corrompida en el camino, no un mensaje de GDELT
                print(f"  respuesta corrompida ({texto[:40]!r}); se reintenta"); fallas_seguidas = 2; continue
            raise RuntimeError(f"GDELT rechazó la consulta (HTTP {codigo}): {texto[:500]}")
    raise RuntimeError("Se agotaron los reintentos. Espere unos minutos y vuelva a ejecutar: la caché conserva lo ya descargado.")

def huella(query):
    return hashlib.sha1(query.encode()).hexdigest()[:12]

def descargar_mes(serie, query, periodo):
    archivo = CACHE / serie / f"{periodo.strftime('%Y%m')}.json"
    if archivo.exists():
        guardado = json.loads(archivo.read_text(encoding="utf-8"))
        if guardado.get("huella_consulta") == huella(query):
            return guardado["respuesta"]
    inicio = periodo.start_time.to_pydatetime()
    fin = periodo.end_time.floor("s").to_pydatetime()
    resp = pedir(query, inicio, fin)
    archivo.parent.mkdir(parents=True, exist_ok=True)
    archivo.write_text(json.dumps({"serie": serie, "consulta": query, "huella_consulta": huella(query), "mes": str(periodo),
                                   "descargado": datetime.now().isoformat(timespec="seconds"), "respuesta": resp}, ensure_ascii=False), encoding="utf-8")
    return resp

def a_diario(resp):
    """Convierte la respuesta de timelinevolraw en filas diarias (fecha, value, norm)."""
    if not resp or "timeline" not in resp or not resp["timeline"]:
        return pd.DataFrame(columns=["fecha", "value", "norm"])
    datos = resp["timeline"][0].get("data", [])
    df = pd.DataFrame(datos)
    if df.empty:
        return pd.DataFrame(columns=["fecha", "value", "norm"])
    df["fecha"] = pd.to_datetime(df["date"].str[:8], format="%Y%m%d")
    return df[["fecha", "value", "norm"]]


if METODO is None:
    probar_conexion()

## 4. Diagnóstico antes de descargar todo
Se prueba cada consulta en **mayo de 2019** (amenaza de aranceles de EE.UU. a México, 30 de mayo), comprobando que:

1. **Qué versión de la consulta acepta GDELT** (se elige la más completa).
2. La respuesta de cada serie es diaria y trae `value` (artículos encontrados) y `norm` (total monitoreado).
3. Si GDELT acepta las consultas solo con país (denominadores) y qué mide `norm`.
4. Cuántos artículos aportarían, como máximo, los términos omitidos.
5. Que los artículos encontrados tengan sentido: se listan 15 títulos de la serie principal.

Si algo falla aquí, **no siga**: revise el mensaje.

In [ ]:
mayo = pd.Period("2019-05", freq="M")
ini_m, fin_m = mayo.start_time.to_pydatetime(), mayo.end_time.floor("s").to_pydatetime()
def diagnosticar(series):
    filas, resp_d = [], {}
    for serie, s_ in series.items():
        try:
            resp = pedir(s_["q"], ini_m, fin_m)
            error = ""
        except RuntimeError as e:
            resp, error = {}, str(e)[:200]
        resp_d[serie] = d = a_diario(resp)
        filas.append(dict(serie=serie, aceptada=error == "", error=error,
                          resolucion=resp.get("query_details", {}).get("date_resolution", "?") if resp else "-",
                          dias=len(d), articulos=int(d.value.sum()) if len(d) else 0, norm_total_mes=int(d.norm.sum()) if len(d) else 0,
                          articulos_30_mayo=int(d.loc[d.fecha == "2019-05-30", "value"].sum()) if len(d) else 0,
                          articulos_31_mayo=int(d.loc[d.fecha == "2019-05-31", "value"].sum()) if len(d) else 0))
    return pd.DataFrame(filas), resp_d

# 4a. Elegir la versión más completa que GDELT acepte (se prueba con la consulta más larga: TPU + México + país)
if NIVEL is None:
    for nivel in NIVELES:
        try:
            pedir(consulta(True, "US", nivel), ini_m, fin_m)
            NIVEL = nivel
            break
        except RuntimeError as e:
            print(f"Versión «{nivel}» rechazada ({len(consulta(True, 'US', nivel))} caracteres): {str(e)[:120]}")
    if NIVEL is None:
        raise RuntimeError("GDELT rechazó todas las versiones de la consulta. Revise el mensaje de error.")
print(f"Versión de la consulta que se usará en todas las series: «{NIVEL}» ({len(consulta(True, 'US'))} caracteres)")
SERIES = construir_series()

# 4b. Probar todas las series y los denominadores en mayo de 2019
diag, respuestas = diagnosticar({**SERIES, **DENOMINADORES})
display(diag)
principales = diag[diag.serie.isin(SERIES)]
assert principales.aceptada.all(), "GDELT rechazó alguna consulta principal: lea la columna 'error'."
assert (principales.resolucion == "day").all() and (principales.dias >= 28).all(), "Alguna consulta principal no devolvió datos diarios completos."

# ¿Qué mide 'norm'? Si es igual con y sin filtro de país, es el total global de GDELT (en las pruebas de octubre de 2026 lo fue).
n_global = respuestas["tpu_mx_global"].set_index("fecha").norm
n_eeuu = respuestas["tpu_mx_eeuu"].set_index("fecha").norm
NORM_ES_GLOBAL = bool(np.allclose(n_global.reindex(n_eeuu.index), n_eeuu))
DENOM_PAIS_OK = bool(diag.loc[diag.serie.isin(DENOMINADORES), "aceptada"].all() and (diag.loc[diag.serie.isin(DENOMINADORES), "articulos"] > 0).all())
if DENOM_PAIS_OK:
    SERIES.update(DENOMINADORES)
    DECISION_DENOM = "Denominador por país: consultas solo con sourcecountry (aceptadas por GDELT)."
elif not NORM_ES_GLOBAL:
    DECISION_DENOM = "GDELT no aceptó las consultas solo con sourcecountry, pero 'norm' cambia con el filtro de país: se usa 'norm' de cada serie como total de su país."
else:
    DECISION_DENOM = ("GDELT no aceptó las consultas solo con sourcecountry y 'norm' es el total global: los índices por país se normalizan "
                      "con el total global (REVISAR: no es el equivalente exacto de N7).")
print(f"'norm' es el total global de GDELT: {NORM_ES_GLOBAL}\nDenominadores por país disponibles: {DENOM_PAIS_OK}\nDecisión: {DECISION_DENOM}")

# 4c. ¿Cuánto se pierde con los términos omitidos? Cada grupo pequeño de términos omitidos se busca por
#     separado (para no pasar el límite de longitud). Es una cota superior: esos artículos pueden
#     coincidir con los que ya se cuentan, así que la pérdida real es menor o igual.
omit_c = [t for t in COMERCIO_COMPLETO + CERCANIA if t not in NIVELES[NIVEL]["comercio"] + NIVELES[NIVEL]["cercania"]]
omit_u = [t for t in INCERTIDUMBRE_COMPLETA if t not in NIVELES[NIVEL]["incertidumbre"]]
base_n = int(respuestas["tpu_mx_eeuu"].value.sum())
perdida = []
trozos = lambda lst, k: [lst[x:x + k] for x in range(0, len(lst), k)]
for tipo, lista in [("comercio", omit_c), ("incertidumbre", omit_u)]:
    for t in trozos(lista, 4):
        q_omit = (f"{grupo(t)} {grupo(['uncertainty', 'uncertain', 'risk', 'risks', 'potential'])}" if tipo == "comercio"
                  else f"{grupo(['tariff', 'tariffs'])} {grupo(t)}") + f" {grupo(MEXICO)} sourcecountry:US"
        try:
            n_ = int(a_diario(pedir(q_omit, ini_m, fin_m)).value.sum())
        except RuntimeError as e:
            n_ = np.nan
        perdida.append(dict(grupo=tipo, terminos_omitidos=", ".join(t), articulos_maximos=n_))
if perdida:
    perdida = pd.DataFrame(perdida)
    display(perdida)
    print(f"Mayo 2019: la serie principal cuenta {base_n:,} artículos. Cada fila muestra cuántos artículos encuentran esos términos omitidos "
          f"combinados con los términos principales del otro grupo. Es una cota superior y muy holgada: la mayoría de esos artículos "
          f"también contiene términos que sí se cuentan (por ejemplo, «risky» casi siempre aparece junto a «risk» o «tariffs»).")
else:
    print("No se omitió ningún término.")

try:
    art = pedir(SERIES["tpu_mx_eeuu"]["q"], pd.Timestamp("2019-05-29").to_pydatetime(), pd.Timestamp("2019-06-02 23:59:59").to_pydatetime(),
                mode="artlist", extra={"maxrecords": 15, "sort": "hybridrel"})
    display(pd.DataFrame(art["articles"])[["seendate", "domain", "title"]] if art.get("articles") else "GDELT no devolvió lista de artículos para este periodo.")
except RuntimeError as e:
    print("No se pudo obtener la lista de artículos de ejemplo:", e)

## 5. Descarga completa (mes por mes, reanudable)

In [ ]:
t0 = time.time()
for serie, s in SERIES.items():
    pendientes = [p for p in MESES if not (CACHE / serie / f"{p.strftime('%Y%m')}.json").exists()]
    print(f"{serie}: {len(MESES) - len(pendientes)} meses ya en caché, {len(pendientes)} por descargar")
    for i, p in enumerate(MESES, 1):
        descargar_mes(serie, s["q"], p)
        if i % 12 == 0:
            print(f"  {serie}: {p} ({(time.time() - t0) / 60:.0f} min)")
print(f"Descarga terminada en {(time.time() - t0) / 60:.1f} minutos.")

## 6. Series mensuales y comprobaciones
Se suman los conteos diarios por mes. Se marca la **completitud**: días del mes con datos (`norm` > 0). Un mes con menos de 90% de días con datos se marca y **no se usa** en el índice (queda como faltante; no se imputa).

In [ ]:
filas = []
for serie, s in SERIES.items():
    for p in MESES:
        d = a_diario(descargar_mes(serie, s["q"], p))
        filas.append(dict(mes=p.to_timestamp(), serie=serie, articulos=d.value.sum() if len(d) else 0,
                          total_gdelt=d.norm.sum() if len(d) else np.nan,
                          dias_con_datos=int((d.norm > 0).sum()) if len(d) else 0, dias_mes=p.days_in_month))
largo = pd.DataFrame(filas)
ancho = largo.pivot(index="mes", columns="serie", values="articulos")
norm_serie = largo.pivot(index="mes", columns="serie", values="total_gdelt")
# Completitud: días con datos según el total monitoreado de la serie global (no depende de que haya artículos encontrados)
ref = largo[largo.serie == "tpu_mx_global"].set_index("mes")
comp = pd.DataFrame({"total_gdelt": ref.total_gdelt, "dias_con_datos": ref.dias_con_datos, "dias_mes": ref.dias_mes})
comp["completitud"] = comp.dias_con_datos / comp.dias_mes
if not DENOM_PAIS_OK:
    if not NORM_ES_GLOBAL:
        ancho["total_eeuu"], ancho["total_mexico"] = norm_serie["tpu_mx_eeuu"], norm_serie["tpu_mx_mexico"]
    else:
        ancho["total_eeuu"] = ancho["total_mexico"] = comp.total_gdelt
mensual = ancho.join(comp)
mensual["mes_completo"] = mensual.completitud >= 0.9

checks = pd.DataFrame([
    dict(prueba="Meses descargados", resultado=f"{len(mensual)} ({mensual.index.min():%Y-%m} a {mensual.index.max():%Y-%m})", estado="OK"),
    dict(prueba="Denominador por país", resultado=DECISION_DENOM, estado="OK" if DENOM_PAIS_OK else "REVISAR"),
    dict(prueba="Meses con al menos 90% de días con datos", resultado=f"{int(mensual.mes_completo.sum())} de {len(mensual)}",
         estado="OK" if mensual.mes_completo.all() else "REVISAR"),
    dict(prueba="Denominadores por país positivos en meses completos",
         resultado=f"EE.UU.: {int((mensual.loc[mensual.mes_completo, 'total_eeuu'] > 0).sum())}; México: {int((mensual.loc[mensual.mes_completo, 'total_mexico'] > 0).sum())}",
         estado="OK" if (mensual.loc[mensual.mes_completo, ["total_eeuu", "total_mexico"]] > 0).all().all() else "ERROR"),
    dict(prueba="El filtro de México nunca da más artículos que la versión sin filtro (EE.UU.)",
         resultado=f"{int((mensual.tpu_mx_eeuu > mensual.tpu_eeuu).sum())} meses con más", estado="OK" if (mensual.tpu_mx_eeuu <= mensual.tpu_eeuu).all() else "ERROR"),
])
display(checks)
if (checks.estado == "ERROR").any():
    raise RuntimeError("Hay comprobaciones en ERROR: revise la tabla antes de continuar.")
display(mensual[~mensual.mes_completo][["dias_con_datos", "dias_mes", "completitud"]])

## 7. Índices
Misma escala que el TPU: 100 = 1% de los artículos.

- `tpu_mx_gdelt_eeuu` = artículos TPU + México en medios de EE.UU. / total de artículos de medios de EE.UU. × 10,000 → **principal**
- `tpu_mx_gdelt_mexico` = lo mismo con medios mexicanos
- `tpu_mx_gdelt_global` = artículos TPU + México en todos los medios / total monitoreado por GDELT × 10,000
- `tpu_gdelt_eeuu` = sin filtro de México, medios de EE.UU. (solo para validar contra Caldara)

In [ ]:
m = mensual.copy()
ok = m.mes_completo
indices = pd.DataFrame(index=m.index)
indices["tpu_mx_gdelt_eeuu"] = (1e4 * m.tpu_mx_eeuu / m.total_eeuu).where(ok)
indices["tpu_mx_gdelt_mexico"] = (1e4 * m.tpu_mx_mexico / m.total_mexico).where(ok)
indices["tpu_mx_gdelt_global"] = (1e4 * m.tpu_mx_global / m.total_gdelt).where(ok)
indices["tpu_gdelt_eeuu"] = (1e4 * m.tpu_eeuu / m.total_eeuu).where(ok)
indices["proporcion_mexico_en_tpu_eeuu"] = (m.tpu_mx_eeuu / m.tpu_eeuu).where(ok)
display(indices.describe().T.round(3))

## 8. Validación contra el TPU de Caldara et al.
Si la traducción del método a GDELT funciona, `tpu_gdelt_eeuu` (sin filtro de México) debe parecerse al TPU original. Se comparan niveles (en logaritmos) y variaciones mensuales, y se grafican las series **estandarizadas** en un mismo eje (no se usan dos ejes).

In [ ]:
# Busca el TPU de Caldara et al. en la carpeta de trabajo; si no está, lo descarga de la página de los autores y lo guarda ahí.
archivo_cal = next((CARPETA / n for n in ["tpu_web_latest.xlsx", "tpu_caldara_iacoviello.xlsx"] if (CARPETA / n).exists()), None)
if archivo_cal is None:
    archivo_cal = CARPETA / "tpu_web_latest.xlsx"
    try:
        r = requests.get(URL_CALDARA, timeout=TIMEOUT_SEG, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        archivo_cal.write_bytes(r.content)
    except Exception as e:
        if not shutil.which("curl"):
            raise RuntimeError(f"No se pudo descargar {URL_CALDARA} ({e}). Descárguelo con el navegador y guárdelo en {CARPETA}.") from e
        subprocess.run(["curl", "-sS", "-L", "-A", "Mozilla/5.0", "-o", str(archivo_cal), URL_CALDARA], check=True, timeout=300)
    print("Descargado:", archivo_cal)
tpu_cal = pd.read_excel(archivo_cal, sheet_name="TPU_MONTHLY")
tpu_cal = tpu_cal.assign(mes=pd.to_datetime(tpu_cal.DATE).dt.to_period("M").dt.to_timestamp()).set_index("mes").TPU
comp_v = pd.concat([np.log(tpu_cal.rename("tpu_caldara")), np.log(indices[["tpu_gdelt_eeuu", "tpu_mx_gdelt_eeuu", "tpu_mx_gdelt_mexico", "tpu_mx_gdelt_global"]])], axis=1, join="inner").dropna()
val = pd.DataFrame({
    "correlación con log TPU Caldara (niveles)": comp_v.corr()["tpu_caldara"],
    "correlación de variaciones mensuales": comp_v.diff().corr()["tpu_caldara"],
}).drop("tpu_caldara").round(3)
display(val)
print(f"Meses comparados: {len(comp_v)} ({comp_v.index.min():%Y-%m} a {comp_v.index.max():%Y-%m})")
if val.loc["tpu_gdelt_eeuu"].iloc[0] < 0.6:
    print("REVISAR: la versión GDELT sin filtro se parece poco al TPU de Caldara (correlación < 0.6). Revise términos y medios antes de usar el índice.")

try:
    import matplotlib.pyplot as plt
    z = (comp_v - comp_v.mean()) / comp_v.std()
    fig, ax = plt.subplots(figsize=(10, 4))
    for c, color, lw in [("tpu_caldara", "#52514e", 1.5), ("tpu_gdelt_eeuu", "#eb6834", 1.5), ("tpu_mx_gdelt_eeuu", "#2a78d6", 2.2)]:
        ax.plot(z.index, z[c], color=color, lw=lw, label={"tpu_caldara": "TPU Caldara et al.", "tpu_gdelt_eeuu": "GDELT, EE.UU., sin filtro México",
                                                           "tpu_mx_gdelt_eeuu": "GDELT, EE.UU., con filtro México"}[c])
    ax.axhline(0, color="#999", lw=0.8); ax.set_ylabel("log, estandarizado"); ax.legend(frameon=False, loc="upper left")
    ax.spines[["top", "right"]].set_visible(False); ax.set_title("Incertidumbre de política comercial: original vs. GDELT")
    plt.tight_layout(); plt.show()
except ImportError:
    print("Instale matplotlib para ver la gráfica (pip install matplotlib).")

## 9. Meses con mayor incertidumbre comercial sobre México
Deben coincidir con episodios conocidos (mayo de 2019, noviembre de 2024, febrero-abril de 2025…).

In [ ]:
display(indices.tpu_mx_gdelt_eeuu.dropna().sort_values(ascending=False).head(12).round(1).rename("tpu_mx_gdelt_eeuu").to_frame()
        .assign(proporcion_mexico=indices.proporcion_mexico_en_tpu_eeuu.round(3)))

## 10. Guardar
Todo en la carpeta de trabajo:
- `tpu_mexico_gdelt_mensual.csv`: conteos, denominadores, completitud e índices por mes.
- `tpu_mexico_gdelt_metadatos.json`: consultas exactas, parámetros y fecha de descarga.
- `gdelt_cache/`: respuestas crudas de GDELT (consérvelas: son la fuente original y permiten reconstruir todo sin volver a descargar).

In [ ]:
salida = mensual.join(indices).reset_index()
salida["mes"] = salida["mes"].dt.strftime("%Y-%m")
salida.to_csv(SALIDA / "tpu_mexico_gdelt_mensual.csv", index=False)
(SALIDA / "tpu_mexico_gdelt_metadatos.json").write_text(json.dumps({
    "fuente": "GDELT DOC 2.0 API, modo timelinevolraw, resolución diaria sumada por mes",
    "url": URL.replace("https://", f"{METODO[1]}://", 1), "cliente": METODO[0], "protocolo": METODO[1], "truststore": _TRUSTSTORE, "meses": [str(MESES[0]), str(MESES[-1])], "pausa_seg": PAUSA_SEG,
    "series": {k: {"consulta": v["q"], "descripcion": v["desc"], "huella": huella(v["q"])} for k, v in SERIES.items()},
    "version_consulta": NIVEL, "terminos": {**NIVELES[NIVEL], "mexico": MEXICO},
    "criterio_mes_completo": "al menos 90% de los días con norm > 0 (serie global)",
    "denominador": DECISION_DENOM, "norm_es_global": NORM_ES_GLOBAL,
    "generado": datetime.now().isoformat(timespec="seconds"), "python": platform.python_version(), "pandas": pd.__version__,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Guardado en", SALIDA)